In [19]:
%cd C:/Users/micha/Documents/master_thesis


import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import random
from torch.utils.tensorboard import SummaryWriter
import os
from datetime import datetime
from tqdm import tqdm
import itertools
import pandas as pd
from sklearn.model_selection import ParameterGrid
import torch.nn.functional as F

from copy import deepcopy
import math
from torch.optim.lr_scheduler import StepLR, ExponentialLR, CosineAnnealingLR, ReduceLROnPlateau, LambdaLR

C:\Users\micha\Documents\master_thesis


In [20]:
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
class Ship:
    def __init__(self, starting_pos, random_start, max_speed, hp, range, stockpile_max):
        self.random_start = random_start
        self.starting_position = starting_pos
        self.starting_hp = hp
        self.position = starting_pos
        self.course = 0
        self.speed = 0
        self.max_speed = max_speed
        self.hp = hp
        self.alive = True
        self.range = range
        self.stockpile_max = stockpile_max
        self.stockpile = stockpile_max
    
    def move(self, action, dt):
        speed_frac, course_scalar = action[:2]
        self.speed  = np.clip(speed_frac, 0.0, 1.0) * self.max_speed
        self.course = (np.clip(course_scalar, -1.0, 1.0) + 1.0) * np.pi
        self.position += np.array([self.speed * np.cos(self.course) * dt, self.speed * np.sin(self.course) * dt])
        np.clip(self.position, -1.0, 1.0, out=self.position)
        
    def take_damage(self, damage):
        self.hp -= damage
        if self.hp <= 0:
            self.alive = False
            
    def reset(self):
        if self.random_start:
            self.position = np.random.uniform(-1.0, 1.0, size=2)
        else:
            self.position = self.starting_position
        self.hp = self.starting_hp
        self.stockpile = self.stockpile_max
        self.course = 0
        self.speed = 0
        self.alive = True

def prime_rare_buffer(agent, env, n_samples=1000):
    for _ in range(n_samples):
        pos_alice = np.array([-0.5, -0.5])
        pos_bob   = np.array([-0.4, -0.5]) 
        state = np.concatenate([pos_alice, pos_bob], dtype=np.float32)

        action = np.array([0.0, 0.0, 0.8, 0.0], dtype=np.float32)

        next_state = state.copy()

        reward = env.cfg_alice.get("victory_bonus", 1.0)

        done = True

        agent.rare_memory_buffer.add(state, action, reward, next_state, done)


def build_scheduler(optimizer, kind: str, **kw):
    kind = kind.lower()

    if kind in ("constant", "fixed", "none"):
        return LambdaLR(optimizer, lr_lambda=lambda step: 1.0)

    if kind == "step":
        return StepLR(optimizer, step_size=kw.get("step_size", 10_000),
                                gamma     =kw.get("gamma",     0.5))

    if kind == "exp":
        return ExponentialLR(optimizer, gamma=kw.get("gamma", 0.99))

    if kind == "cosine":
        return CosineAnnealingLR(optimizer,
                                T_max = kw["T_max"],
                                eta_min = kw.get("eta_min", 1e-6))

    if kind == "plateau":
        return ReduceLROnPlateau(optimizer,
                                mode      = kw.get("mode", "min"),
                                patience  = kw.get("patience", 20_000),
                                factor    = kw.get("factor", 0.5),
                                min_lr    = kw.get("min_lr", 1e-6))
    return None

class BaseNoise:
    def reset(self): 
        pass

    def __call__(self):
        raise NotImplementedError

class GaussianNoise(BaseNoise):
    def __init__(self, dim, sigma):
        self.dim   = dim
        self.sigma = sigma

    def __call__(self):
        return self.sigma * np.random.randn(self.dim)

class OUNoise(BaseNoise):
    def __init__(self, dim, theta=0.15, sigma=0.2, dt=1e-2):
        self.dim   = dim
        self.theta = theta
        self.sigma = sigma
        self.dt    = dt
        self.reset()

    def reset(self):
        self.state = np.zeros(self.dim, dtype=np.float32)

    def __call__(self):
        dx = self.theta * (-self.state) * self.dt \
             + self.sigma * np.sqrt(self.dt) * np.random.randn(self.dim)
        self.state += dx
        return self.state

class EpsilonGreedyNoise(BaseNoise):
    def __init__(self, dim, epsilon):
        self.dim     = dim
        self.epsilon = epsilon
        
    def __call__(self):
        if np.random.rand() < self.epsilon:
            return np.random.uniform(-1.0, 1.0, size=self.dim)
        return np.zeros(self.dim, dtype=np.float32)

class SparseWeaponNoise(BaseNoise):
    def __init__(self, n_ships, p_fire=0.05, fire_low=0.6, fire_high=1.0):
        self.n_ships   = n_ships
        self.p_fire    = p_fire
        self.fire_low  = fire_low
        self.fire_high = fire_high
        self.dim       = 4 * n_ships

    def __call__(self):
        noise = np.zeros(self.dim, dtype=np.float32)

        for k in range(self.n_ships):
            base = 4 * k

            if np.random.rand() < self.p_fire:
                noise[base + 2] = np.random.uniform(self.fire_low, self.fire_high)
                noise[base + 3] = np.random.uniform(-1.0, 1.0)

        return noise

class CompositeNoise(BaseNoise):
    def __init__(self, *sources):
        assert sources, "need at least one noise source"
        self.sources = sources
        self.dim     = sources[0].dim
        # sanity check
        for s in self.sources:
            if s.dim != self.dim:
                raise ValueError("all noise sources must share the same `dim`")

    def reset(self):
        for s in self.sources:
            if hasattr(s, "reset"):
                s.reset()

    def __call__(self):
        out = np.zeros(self.dim, dtype=np.float32)
        for s in self.sources:
            out += s()
        return out

def make_noise(kind, dim=None, **kw):
    kind = kind.lower()
    
    if kind in ("decay", "annealed", "exp-decay"):
        # build the *base* noise first
        base_spec = kw["base"]
        base_kind = base_spec["kind"]
        base_dim  = base_spec.get("dim", dim)
        base_kw   = {k: v for k, v in base_spec.items()
                            if k not in ("kind", "dim")}
        base      = make_noise(base_kind, base_dim, **base_kw)

        return ExpDecayNoise(
            base_noise = base,
            scale_init = kw.get("scale_init", 1.0),
            gamma      = kw.get("gamma",      0.999),
            scale_min  = kw.get("scale_min",  0.05),
            per        = kw.get("per",        "step")
            )
    if kind in ("gauss", "gaussian", "normal"):
        return GaussianNoise(dim, sigma=kw.get("sigma", 0.1))

    if kind in ("ou", "ornstein-uhlenbeck"):
        return OUNoise(dim,
                        theta = kw.get("theta", 0.15),
                        sigma = kw.get("sigma", 0.2),
                        dt    = kw.get("dt",    1e-2))

    if kind in ("eps", "epsilon", "epsilon-greedy"):
        return EpsilonGreedyNoise(dim, epsilon=kw.get("epsilon", 0.1))

    if kind in ("sparse", "sparse-weapon", "weapon"):
        n_ships    = kw.pop("n_ships")
        p_fire     = kw.pop("p_fire",     0.05)
        fire_low   = kw.pop("fire_low",   0.6)
        fire_high  = kw.pop("fire_high",  1.0)
        return SparseWeaponNoise(n_ships,
                                p_fire=p_fire,
                                fire_low=fire_low,
                                fire_high=fire_high)

    if kind in ("combo", "composite"):
        parts = kw["parts"]
        sources = []
        for spec in parts:
            k = spec["kind"]
            src_dim = spec.get("dim", dim)
            spec_kw = {kk: vv for kk, vv in spec.items() if kk != "kind"}
            sources.append(make_noise(k, src_dim, **spec_kw))
        return CompositeNoise(*sources)

    raise ValueError(f"Unknown noise kind '{kind}'")

class ExpDecayNoise(BaseNoise):
    def __init__(self, base_noise, scale_init=1.0,
                    gamma=0.999, scale_min=0.05, per="step"):
        self.n         = base_noise
        self.scale     = scale_init
        self.gamma     = gamma
        self.scale_min = scale_min
        assert per in ("step", "episode")
        self.per       = per
        self.dim       = self.n.dim

    def reset(self):
        self.n.reset()
        if self.per == "episode":
            self.scale = max(self.scale * self.gamma, self.scale_min)

    def __call__(self):
        out = self.n() * self.scale
        if self.per == "step":
            self.scale = max(self.scale * self.gamma, self.scale_min)
        return out

class RunningMeanStd:
    def __init__(self, shape, epsilon: float = 1e-4, dtype=np.float64):
        self.mean  = np.zeros(shape, dtype=dtype)
        self.var   = np.ones(shape,  dtype=dtype)
        self.count = epsilon                      # protects against /0

    # ------------------------------------------------------------------ #
    def update(self, x: np.ndarray):
        x          = np.asarray(x, dtype=self.mean.dtype)
        batch_mean = x.mean(axis=0)
        batch_var  = x.var(axis=0)
        batch_cnt  = x.shape[0]

        delta       = batch_mean - self.mean
        tot_count   = self.count + batch_cnt

        # updated mean
        new_mean    = self.mean + delta * batch_cnt / tot_count

        # pooled variance
        m_a = self.var * self.count
        m_b = batch_var * batch_cnt
        m2  = m_a + m_b + delta**2 * self.count * batch_cnt / tot_count
        new_var = m2 / tot_count

        # assign back
        self.mean   = new_mean
        self.var    = new_var
        self.count  = tot_count

    def normalize(self, x, clip=None):
        """
        Return normalised copy of `x`.
        If `clip` is set, values are clipped to [-clip, clip] after scaling.
        """
        x = (x - self.mean) / (np.sqrt(self.var) + 1e-8)
        if clip is not None:
            x = np.clip(x, -clip, clip)
        return x

    @property
    def std(self):
        return np.sqrt(self.var)
    
def scalar_to_intshots(f, max_shots):
    return int(np.clip(np.round(f * max_shots), 0, max_shots))

def scalar_to_index(u, n):
    return int(np.clip(np.round(0.5 * (u + 1) * (n - 1)), 0, n - 1))

In [ ]:
class NavalEnvironment(gym.Env):
    def __init__(self, fleet_alice, fleet_bob, max_steps, agent_cfgs, meet_dist = 0.1, dt = 0.05, max_shots = 3):
        super().__init__()
        self.fleet_alice   = fleet_alice
        self.fleet_bob     = fleet_bob
        self.max_steps  = max_steps
        self.step_count = 0
        self.meet_dist  = meet_dist
        self.dt         = dt
        self.n_alice    = len(fleet_alice)
        self.n_bob      = len(fleet_bob)
        self.max_shots = max_shots
        self.fire_threshold = 0.5 
        self.rare_event = False
        self.rare_event_attempt = False
        
        # list of two dicts, one per agent
        self.cfg_alice  = agent_cfgs[0]
        self.cfg_bob    = agent_cfgs[1]
        
        low_pr_ship = [-1.0, 0.0, 0, -1]
        high_pr_ship = [1.0, 1.0, 1, 1]
        
        # Define spaces
        action_low  = np.tile(low_pr_ship, self.n_alice + self.n_bob).astype(np.float32)
        action_high = np.tile(high_pr_ship, self.n_alice + self.n_bob).astype(np.float32)
        self.action_space = gym.spaces.Box(action_low, action_high, dtype=np.float32)

        self.state_space = gym.spaces.Box(0.0, 1.0, shape=(2 * (self.n_alice + self.n_bob),), dtype=np.float32)

    def reset(self, random_start=False):
        for ship in self.fleet_alice + self.fleet_bob:
            ship.reset()
        self.step_count = 0
        return self._get_state()

    def _get_state(self):
        pos_alice = np.stack([ship.position for ship in self.fleet_alice])  # (nA,2)
        pos_bob   = np.stack([ship.position for ship in self.fleet_bob])    # (nB,2)
        return np.concatenate([pos_alice.ravel(), pos_bob.ravel()]).astype(np.float32)

    def step(self, action):
        self.rare_event = False
        self.rare_event_attempt = False
        
        # Unpack action vector
        action = action.reshape(-1, 4)
        action_alice = action[:self.n_alice]
        action_bob   = action[self.n_alice:]
        
        # Movement
        for ship, move_order in zip(self.fleet_alice, action_alice):
            if ship.alive:
                ship.move(move_order[:2], self.dt)

        for ship, move_order in zip(self.fleet_bob, action_bob):
            if ship.alive:
                ship.move(move_order[:2], self.dt)

        self.step_count += 1

        # Update state
        pos_alice = np.stack([s.position for s in self.fleet_alice])
        pos_bob   = np.stack([s.position for s in self.fleet_bob])
        distance_matrix = np.linalg.norm(pos_alice[:, None, :] - pos_bob[None, :, :], axis=-1)


        # Rewards
        # Attractive term to closest opponent
        r_ship_A = -distance_matrix.sum(axis=1) * self.cfg_alice.get("r_attract")
        r_ship_B = -distance_matrix.sum(axis=0) * self.cfg_bob.get("r_attract")

        # Wall penalty
        wall_th_A = self.cfg_alice.get("wall_threshold", 0.0)
        if wall_th_A > 0:
            dist_wall_A = np.minimum.reduce([np.abs(pos_alice[:,0] - 1),
                                            np.abs(pos_alice[:,0] + 1),
                                            np.abs(pos_alice[:,1] - 1),
                                            np.abs(pos_alice[:,1] + 1)])
            viol_A = np.clip(wall_th_A - dist_wall_A, 0.0, None)
            r_ship_A += viol_A * self.cfg_alice.get("wall_penalty", 0.0)

        wall_th_B = self.cfg_bob.get("wall_threshold", 0.0)
        if wall_th_B > 0:
            dist_wall_B = np.minimum.reduce([np.abs(pos_bob[:,0] - 1),
                                            np.abs(pos_bob[:,0] + 1),
                                            np.abs(pos_bob[:,1] - 1),
                                            np.abs(pos_bob[:,1] + 1)])
            viol_B = np.clip(wall_th_B - dist_wall_B, 0.0, None)
            r_ship_B += viol_B * self.cfg_bob.get("wall_penalty", 0.0)

        # Time penalty
        r_ship_A += self.cfg_alice.get("time_penalty", 0.0)
        r_ship_B += self.cfg_bob.get("time_penalty", 0.0)

        reward_A = r_ship_A.mean()
        reward_B = r_ship_B.mean()
        
        # Firing mechanics
        pending_damage_alice = np.zeros(self.n_bob, dtype=np.float32)
        pending_damage_bob   = np.zeros(self.n_alice, dtype=np.float32)
        
        for i, ship in enumerate(self.fleet_alice):
            if ship.alive and ship.stockpile > 0:
                fire_frac = action_alice[i, 2]
                if fire_frac > self.fire_threshold:
                    shots_to_fire = scalar_to_intshots(fire_frac, self.max_shots)
                    shots = min(shots_to_fire, ship.stockpile)
                    reward_A += self.cfg_alice.get("reward_fire", 0.0)
                    self.rare_event_attempt = True
                    if shots > 0:
                        j     = scalar_to_index(action_alice[i, 3], self.n_bob)
                        target  = self.fleet_bob[j]
                        if target.alive and target.hp > 0:
                            if distance_matrix[i, j] <= ship.range:
                                pending_damage_alice += shots
                                ship.stockpile -= shots
                                self.rare_event = True
                                #print(f"Alice ship {i} fired {shots} shots at Bob ship {j}")

        for i, ship in enumerate(self.fleet_bob):
            if ship.alive and ship.stockpile > 0:
                fire_frac = action_bob[i, 2]
                if fire_frac > self.fire_threshold:
                    shots_to_fire = scalar_to_intshots(fire_frac, self.max_shots)
                    shots = min(shots_to_fire, ship.stockpile)
                    reward_B += self.cfg_bob.get("reward_fire", 0.0)
                    self.rare_event_attempt = True
                    if shots > 0:
                        j     = scalar_to_index(action_bob[i, 3], self.n_alice)
                        target  = self.fleet_alice[j]
                        if target.alive and target.hp > 0:
                            if distance_matrix[i, j] <= ship.range:
                                pending_damage_bob += shots
                                ship.stockpile -= shots
                                self.rare_event = True
                                #print(f"Bob ship {i} fired {shots} shots at Alice ship {j}")

        for j, dmg in enumerate(pending_damage_bob):
            if dmg > 0:
                self.fleet_bob[j].take_damage(dmg)

        for i, dmg in enumerate(pending_damage_alice):
            if dmg > 0:
                self.fleet_alice[i].take_damage(dmg)
                                
        rewards  = np.array([reward_A, reward_B], dtype=np.float32)
        
        alive_alice = [s.alive for s in self.fleet_alice]
        alive_bob = [s.alive for s in self.fleet_bob]
        
        fleet_alice_dead = not any(alive_alice)
        fleet_bob_dead = not any(alive_bob)

        done = (
            fleet_alice_dead or
            fleet_bob_dead or
            (self.step_count >= self.max_steps)
        )

        if done:
            if fleet_alice_dead:
                rewards[0] += self.cfg_alice.get("victory_bonus", 5.0)
            if fleet_bob_dead:
                rewards[1] += self.cfg_bob.get("victory_bonus", 5.0)

        return self._get_state(), rewards, done, {}


In [ ]:
class Actor(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, 128), 
            nn.ReLU(),
            nn.Linear(128, 128),       
            nn.ReLU(),
            nn.Linear(128, 128), 
            nn.ReLU(),
            nn.Linear(128, 128), 
            nn.ReLU(),
            nn.Linear(128, action_dim), 
            nn.Tanh()
        )
        
    def forward(self, x):
        return self.net(x)

class Critic(nn.Module):
    def __init__(self, state_dim, action_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim + action_dim, 128), 
            nn.ReLU(),
            nn.Linear(128, 128),       
            nn.ReLU(),
            nn.Linear(128, 128), 
            nn.ReLU(),
            nn.Linear(128, 128), 
            nn.ReLU(),
            nn.Linear(128, 1)
        )
        
    def forward(self, s, a):
        return self.net(torch.cat([s, a], dim=-1))

class MemoryBuffer:
    def __init__(self, capacity=100_000):
        self.buffer = deque(maxlen=capacity)
        
    def add(self, s, a, r, ns, d):
        self.buffer.append((s, a, r, ns, d))
        
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return (
            torch.as_tensor(np.array(s,  dtype=np.float32)),
            torch.as_tensor(np.array(a,  dtype=np.float32)),
            torch.as_tensor(np.array(r,  dtype=np.float32).reshape(-1, 1)),
            torch.as_tensor(np.array(ns, dtype=np.float32)),
            torch.as_tensor(np.array(d,  dtype=np.float32).reshape(-1, 1))
        )
        
    def __len__(self):
        return len(self.buffer)

class DDPGAgent:
    def __init__(self, state_dim, action_dim, lr_actor, lr_critic, gamma, tau, schedule_actor, schedule_critic, noise_cfg):
        self.actor = Actor(state_dim, action_dim)
        self.actor_target = deepcopy(self.actor)

        self.critic1 = Critic(state_dim, action_dim)
        self.critic1_target = deepcopy(self.critic1)

        self.critic2 = Critic(state_dim, action_dim)
        self.critic2_target = deepcopy(self.critic2)

        self.actor_optim = torch.optim.Adam(self.actor.parameters(), lr=lr_actor)
        self.critic1_optim = torch.optim.Adam(self.critic1.parameters(), lr=lr_critic)
        self.critic2_optim = torch.optim.Adam(self.critic2.parameters(), lr=lr_critic)

        self.actor_scheduler = build_scheduler(self.actor_optim, **schedule_actor) if schedule_actor else None
        self.critic_scheduler = build_scheduler(self.critic1_optim, **schedule_critic) if schedule_critic else None

        self.noise = make_noise(**noise_cfg) if noise_cfg else make_noise("gauss", action_dim, sigma=0.1)

        self.policy_delay = 2
        self.policy_noise = 0.1
        self.noise_clip = 0.2
        self.train_step = 0

        self.main_memory_buffer = MemoryBuffer()
        self.rare_memory_buffer = MemoryBuffer()

        self.gamma = gamma
        self.tau = tau

        self.last_actor_loss = 0.0
        self.last_critic_loss = 0.0
        self.last_q_value = 0.0

    def _safe_sample(self, buffer: MemoryBuffer, k: int):
        if len(buffer) >= k:
            return buffer.sample(k)

        # draw with replacement
        idxs = np.random.randint(0, len(buffer), size=k)
        s, a, r, ns, d = zip(*(buffer.buffer[i] for i in idxs))
        return (
            torch.as_tensor(np.array(s , dtype=np.float32)),
            torch.as_tensor(np.array(a , dtype=np.float32)),
            torch.as_tensor(np.array(r , dtype=np.float32).reshape(-1, 1)),
            torch.as_tensor(np.array(ns, dtype=np.float32)),
            torch.as_tensor(np.array(d , dtype=np.float32).reshape(-1, 1)),
        )

    def store_transition(self, s, a, r, ns, d, rare_flag=False):
        self.main_memory_buffer.add(s, a, r, ns, d)
        if rare_flag:
            self.rare_memory_buffer.add(s, a, r, ns, d)

    def _sample_exact(self, buffer: MemoryBuffer, k: int):
        if len(buffer) >= k:
            return buffer.sample(k)
        idx = np.random.randint(0, len(buffer), size=k)
        s, a, r, ns, d = zip(*(buffer.buffer[i] for i in idx))
        return (
            torch.as_tensor(np.array(s , dtype=np.float32)),
            torch.as_tensor(np.array(a , dtype=np.float32)),
            torch.as_tensor(np.array(r , dtype=np.float32).reshape(-1, 1)),
            torch.as_tensor(np.array(ns, dtype=np.float32)),
            torch.as_tensor(np.array(d , dtype=np.float32).reshape(-1, 1)),
        )

    def _want_attempts(self, attempt_rare_transition):
        return len(self.rare_memory_buffer) < attempt_rare_transition

    def select_action(self, state):
        state_t = torch.tensor(state, dtype=torch.float32).unsqueeze(0)
        action = self.actor(state_t).detach().numpy()[0]
        noise = self.noise()
        action += noise
        self.last_noise = noise
        return np.clip(action, -1.0, 1.0)

    def train(self, batch_size=64, rare_ratio=0.1):
        if len(self.main_memory_buffer) < batch_size:
            return

        k_target = int(batch_size * rare_ratio)
        fill_frac = len(self.rare_memory_buffer) / max(k_target, 1)
        k_rare = min(k_target, int(k_target * fill_frac))
        k_main = batch_size - k_rare

        main_batch = self._sample_exact(self.main_memory_buffer, k_main)
        if k_rare:
            rare_batch = self._sample_exact(self.rare_memory_buffer, k_rare)
            states = torch.cat([main_batch[0], rare_batch[0]], dim=0)
            actions = torch.cat([main_batch[1], rare_batch[1]], dim=0)
            rewards = torch.cat([main_batch[2], rare_batch[2]], dim=0)
            next_s = torch.cat([main_batch[3], rare_batch[3]], dim=0)
            dones = torch.cat([main_batch[4], rare_batch[4]], dim=0)
        else:
            states, actions, rewards, next_s, dones = main_batch

        perm = torch.randperm(batch_size)
        states, actions, rewards, next_s, dones = \
            states[perm], actions[perm], rewards[perm], next_s[perm], dones[perm]

        with torch.no_grad():
            noise = (torch.randn_like(actions) * self.policy_noise).clamp(-self.noise_clip, self.noise_clip)
            next_actions = (self.actor_target(next_s) + noise).clamp(-1.0, 1.0)
            q1_next = self.critic1_target(next_s, next_actions)
            q2_next = self.critic2_target(next_s, next_actions)
            target_q = rewards + self.gamma * (1 - dones) * torch.min(q1_next, q2_next)

        current_q1 = self.critic1(states, actions)
        current_q2 = self.critic2(states, actions)
        critic1_loss = F.smooth_l1_loss(current_q1, target_q)
        critic2_loss = F.smooth_l1_loss(current_q2, target_q)

        self.critic1_optim.zero_grad()
        critic1_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.critic1.parameters(), max_norm=1.0)
        self.critic1_optim.step()

        self.critic2_optim.zero_grad()
        critic2_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.critic2.parameters(), max_norm=1.0)
        self.critic2_optim.step()

        if self.critic_scheduler is not None:
            self.critic_scheduler.step(critic1_loss.item() if isinstance(self.critic_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau) else None)

        self.last_critic_loss = (critic1_loss.item() + critic2_loss.item()) / 2
        self.last_q_value = current_q1.mean().item()

        if self.train_step % self.policy_delay == 0:
            policy_actions = self.actor(states)
            action_reg = (policy_actions**2).mean()  # regularization term
            actor_loss = -self.critic1(states, policy_actions).mean() + 1e-3 * action_reg
            self.actor_optim.zero_grad()
            actor_loss.backward()
            torch.nn.utils.clip_grad_norm_(self.actor.parameters(), max_norm=1.0)
            self.actor_optim.step()

            if self.actor_scheduler is not None:
                self.actor_scheduler.step(actor_loss.item() if isinstance(self.actor_scheduler, torch.optim.lr_scheduler.ReduceLROnPlateau) else None)

            self.last_actor_loss = actor_loss.item()

            self.soft_update(self.actor_target, self.actor, self.tau)
            self.soft_update(self.critic1_target, self.critic1, self.tau)
            self.soft_update(self.critic2_target, self.critic2, self.tau)

        self.train_step += 1

    def soft_update(self, target, source, tau):
        for t_param, s_param in zip(target.parameters(), source.parameters()):
            t_param.data.copy_(tau * s_param.data + (1 - tau) * t_param.data)


In [24]:
# Agent configurations
cfg_alice = {
    #Rewards
    "r_attract":            0.05,   # multiplies the −d term
    
    "time_penalty":         -0.01, # per‐step cost
    "victory_bonus":        10.0,   # when they come within meet_dist
    "time_up_penalty":      -10.0,  # when max_steps reached
    "wall_threshold":       0.05,  # distance from any wall to start penalizing
    "wall_penalty":         -2.0,  # violation penalty
    "reward_fire":          0.0,  # reward for firing
}

cfg_bob = {
    #Rewards
    "r_attract":            0.05,
    
    "time_penalty":         -0.01,
    "victory_bonus":        10.0,
    "time_up_penalty":      -10.0,
    "wall_threshold":       0.05,
    "wall_penalty":         -2.0,
    "reward_fire":          0.0,
}

# bundle them into a list (or tuple)
agent_cfgs = [cfg_alice, cfg_bob]

In [25]:
fleet_alice =   [Ship(starting_pos=[-0.6,-0.6], random_start = False,
                    max_speed = 1, hp = 1.0, range = 0.3, stockpile_max = 3.0)]

fleet_bob =     [Ship(starting_pos=[0.6,0.6], random_start = False,
                    max_speed = 1, hp = 1.0, range = 0.3, stockpile_max = 3.0)]

In [ ]:
# Main script ______________________________________________________
def training_loop(lr_actor, lr_critic, gamma, tau, batch_size,
                    num_episodes = 1000, warmup = 2000, 
                    attempt_rare_transition = 1000,
                    normalize = True, agent_cfgs = agent_cfgs, 
                    greedy_rollout = True, num_rollouts = 5, log_dir = None):
    
    if log_dir is None:
        ts      = datetime.now().strftime("%Y%m%d_%H%M%S")
        log_dir = os.path.join("runs", f"chase_default_{ts}")
    os.makedirs(log_dir, exist_ok=True)
    writer = SummaryWriter(log_dir=log_dir)
    
    #Initialize environment and agents
    env             = NavalEnvironment(fleet_alice, fleet_bob, max_steps=100, agent_cfgs=agent_cfgs)
    # Extract dimensions of state and action spaces
    state_dim       = env.state_space.shape[0]
    
    commander_alice = DDPGAgent(state_dim = env.state_space.shape[0], action_dim = 4 * len(fleet_alice),
                            lr_actor = lr_actor, lr_critic = lr_critic,
                            gamma = gamma, tau = tau,
                            schedule_actor = dict(kind="constant"),
                            schedule_critic = dict(kind="constant"),
                            noise_cfg = dict(kind = "composite",
                                            dim = 4 * len(fleet_alice),
                                            parts = [
                                                dict(kind="decay", per="episode", gamma=0.995, scale_min=0.02,
                                                            base=dict(kind="gauss", sigma=0.2)),        # decayed movement noise
                                                dict(kind="sparse-weapon", n_ships=len(fleet_alice), p_fire=0.5)  # constant spike strength
                                            ]))


    commander_bob   = DDPGAgent(state_dim = env.state_space.shape[0], action_dim = 4 * len(fleet_bob),
                            lr_actor = lr_actor, lr_critic = lr_critic,
                            gamma = gamma, tau = tau,
                            schedule_actor = dict(kind="constant"),
                            schedule_critic = dict(kind="constant"),
                            noise_cfg = dict(kind = "composite",
                                            dim = 4 * len(fleet_bob),
                                            parts = [
                                                dict(kind="decay", per="episode", gamma=0.995, scale_min=0.02,
                                                            base=dict(kind="gauss", sigma=0.2)),        # decayed movement noise
                                                dict(kind="sparse-weapon", n_ships=len(fleet_alice), p_fire=0.5)  # constant spike strength
                                            ]))
    
    prime_rare_buffer(commander_alice, env, n_samples = 500)
    prime_rare_buffer(commander_alice, env, n_samples = 500)
    action_dim      = env.action_space.shape[0]

    # Initialize running mean and std for state and reward normalization
    state_rms               = RunningMeanStd(shape=(state_dim,))
    reward_rms_alice        = RunningMeanStd(shape=())
    reward_rms_bob          = RunningMeanStd(shape=())

    # Training loop
    all_trajectories = []
    episode_rewards = []
    episode_lengths = []
    dist_alice = np.zeros(len(env.fleet_alice), dtype=np.float32)
    dist_bob = np.zeros(len(env.fleet_bob),   dtype=np.float32)
    
    ep = 0
    global_step = 0
    
    writer.add_scalar("alice/actor_loss",  0.0, global_step)
    writer.add_scalar("alice/critic_loss", 0.0, global_step)
    writer.add_scalar("alice/q_value",     0.0, global_step)
    writer.add_scalar("alice/step_reward", 0.0, global_step)
    writer.add_scalar("bob/actor_loss",    0.0, global_step)
    writer.add_scalar("bob/critic_loss",   0.0, global_step)
    writer.add_scalar("bob/q_value",       0.0, global_step)
    writer.add_scalar("bob/step_reward",   0.0, global_step)
    rare_event_window = deque(maxlen=100)
    rare_attempt_window = deque(maxlen=100)
    trails_every_x_episodes = int(num_episodes // num_rollouts)
    for ep in tqdm(range(num_episodes)):
        raw_state     = env.reset()
        
        p0, p_final, decay_eps = 0.80, 0.05, 1000
        new_p = max(p_final, p0 * (1 - ep / decay_eps))

        weapon_noise_A = commander_alice.noise.sources[1]
        weapon_noise_B = commander_bob.noise.sources[1]


        weapon_noise_A.p_fire = new_p
        weapon_noise_B.p_fire = new_p

        writer.add_scalar("noise/p_fire", new_p, ep)

        
        if normalize and global_step > warmup:
            state_rms.update(raw_state[None, :])
            state = state_rms.normalize(raw_state)
        else:
            state = raw_state
        

        dist_A = np.zeros(len(env.fleet_alice), dtype=np.float32)
        dist_B = np.zeros(len(env.fleet_bob),   dtype=np.float32)

        done      = False                   
        ep_reward = np.array([0.0, 0.0])    
        step      = 0              
        prev_pos_alice = np.stack([s.position.copy() for s in env.fleet_alice])
        prev_pos_bob = np.stack([s.position.copy() for s in env.fleet_bob])

        while not done:
            if global_step == warmup + 1:
                print(f"Warmup complete, beginning training...")

            action_alice = commander_alice.select_action(state)
            action_bob = commander_bob.select_action(state)

            action = np.concatenate([action_alice, action_bob])

            # Noise logs
            noise_rms_A = float(np.sqrt(np.mean(commander_alice.last_noise ** 2)))
            noise_rms_B = float(np.sqrt(np.mean(commander_bob.last_noise  ** 2)))
            
            writer.add_scalar("alice/noise_rms",  noise_rms_A, global_step)
            writer.add_scalar("bob/noise_rms",    noise_rms_B, global_step)
            
            # Execute actions in the environment 
            next_raw_state, raw_reward, done, _ = env.step(action)
            ep_reward += raw_reward
            
            # Normalize state and reward
            if normalize and global_step > warmup:
                state_rms.update(next_raw_state[None, :])
                next_state = state_rms.normalize(next_raw_state) 

                reward_rms_alice.update(np.array([raw_reward[0]])) 
                reward_alice = reward_rms_alice.normalize(np.array([raw_reward[0]]))[0]
                reward_alice = np.clip(reward_alice, -1.0, 1.0)

                reward_rms_bob.update(np.array([raw_reward[1]])) 
                reward_bob   = reward_rms_bob.normalize(np.array([raw_reward[1]]))[0]
                reward_bob   = np.clip(reward_bob,   -1.0, 1.0)
            else:
                next_state = next_raw_state
                reward_alice = raw_reward[0]
                reward_bob   = raw_reward[1]
            
            if commander_alice._want_attempts(attempt_rare_transition): 
                rare_flag = env.rare_event
            else:                                     
                rare_flag = env.rare_event
            
            commander_alice.store_transition(state, action_alice,
                                    reward_alice, next_state, done,
                                    rare_flag=rare_flag)
            
            commander_bob.store_transition(state, action_bob,
                                    reward_bob, next_state, done,
                                    rare_flag=rare_flag)
            
            # Train both agents by sampling from their memory buffers
            if global_step > warmup:
                commander_alice.train(batch_size=batch_size)
                commander_bob.train(batch_size=batch_size)

            # Log metrics to TensorBoard
            if global_step > warmup:
                writer.add_scalar("alice/step_reward",  float(raw_reward[0]),  global_step)
                writer.add_scalar("alice/q_value",      float(commander_alice.last_q_value), global_step)
                writer.add_scalar("alice/actor_loss",   float(commander_alice.last_actor_loss), global_step)
                writer.add_scalar("alice/critic_loss",  float(commander_alice.last_critic_loss), global_step)
                writer.add_scalar("bob/step_reward",    float(raw_reward[1]),  global_step)
                writer.add_scalar("bob/q_value",        float(commander_bob.last_q_value), global_step)
                writer.add_scalar("bob/actor_loss",     float(commander_bob.last_actor_loss), global_step)
                writer.add_scalar("bob/critic_loss",    float(commander_bob.last_critic_loss), global_step)

            # Update distance travelled this step
            pos_alice = np.stack([s.position for s in env.fleet_alice])
            pos_bob   = np.stack([s.position for s in env.fleet_bob])

            step_dist_alice = np.linalg.norm(pos_alice - prev_pos_alice, axis=1)  # (nA,)
            step_dist_bob   = np.linalg.norm(pos_bob   - prev_pos_bob,   axis=1)  # (nB,)

            dist_A += step_dist_alice
            dist_B += step_dist_bob

            prev_pos_alice, prev_pos_bob = pos_alice, pos_bob

            state = next_state
            global_step += 1
            step  += 1
        
        if hasattr(commander_alice.noise, "scale"):
            writer.add_scalar("alice/noise_scale", float(commander_alice.noise.scale), ep)
        if hasattr(commander_bob.noise, "scale"):
            writer.add_scalar("bob/noise_scale",   float(commander_bob.noise.scale), ep)


        # Episode metrics to TensorBoard
        writer.add_scalar("alice/cumulative_episode_reward", float(ep_reward[0]), ep)
        writer.add_scalar("bob/cumulative_episode_reward",   float(ep_reward[1]), ep)
        writer.add_scalar("overall/episode length",          float(step),         ep)
        
        # per-ship episode metrics 
        avg_speed_A = dist_A / (step * env.dt + 1e-9)
        avg_speed_B = dist_B / (step * env.dt + 1e-9)

        for i, (v, d) in enumerate(zip(avg_speed_A, dist_A)):
            writer.add_scalar(f"alice/ship{i}_avg_speed",   float(v), ep)
            writer.add_scalar(f"alice/ship{i}_distance",    float(d), ep)

        for i, (v, d) in enumerate(zip(avg_speed_B, dist_B)):
            writer.add_scalar(f"bob/ship{i}_avg_speed",     float(v), ep)
            writer.add_scalar(f"bob/ship{i}_distance",      float(d), ep)

        rare_event_occurred = int(env.rare_event)
        rare_event_attempted = int(env.rare_event_attempt)

        rare_event_window.append(rare_event_occurred)
        rare_attempt_window.append(rare_event_attempted)

        rare_event_rate = sum(rare_event_window) / len(rare_event_window)
        rare_attempt_rate = sum(rare_attempt_window) / len(rare_attempt_window)

        writer.add_scalar("env/rare_event_avg",    rare_event_rate, ep)
        writer.add_scalar("env/rare_attempt_avg",  rare_attempt_rate, ep)
        writer.add_scalar("env/rare_event_occurred", rare_event_occurred, ep)
        writer.add_scalar("env/rare_event_attempted", rare_event_attempted, ep)

        
        episode_rewards.append(ep_reward.copy())
        episode_lengths.append(step)
        ep = ep + 1

        # every X episodes, do a greedy rollout 
        if greedy_rollout and (ep + 1) % trails_every_x_episodes == 0:

            traj_A, traj_B = [], []                 

            eval_state = env.reset()
            eval_done  = False

            while not eval_done:
                traj_A.append(np.stack([s.position.copy() for s in env.fleet_alice]))
                traj_B.append(np.stack([s.position.copy() for s in env.fleet_bob]))

                # deterministic actions
                a_A = commander_alice.select_action(eval_state)
                a_B = commander_bob.select_action(eval_state)
                eval_state, _, eval_done, _ = env.step(np.concatenate([a_A, a_B]))

            # final point after the last env.step
            traj_A.append(np.stack([s.position.copy() for s in env.fleet_alice]))
            traj_B.append(np.stack([s.position.copy() for s in env.fleet_bob]))

            all_trajectories.append({
                "episode": ep + 1,
                "alice":   np.array(traj_A, dtype=np.float32),
                "bob":     np.array(traj_B, dtype=np.float32)
            })
            print(f"→ Captured trajectories at episode {ep+1} (len={len(traj_A)})")

    
    if greedy_rollout:
        traj_arr = np.array(all_trajectories, dtype=object)
        np.save("trajectories.npy", traj_arr, allow_pickle=True)
        print(f"Saved {len(all_trajectories)} trajectory pairs to 'trajectories.npy'")
    
    ep_arr = np.stack(episode_rewards, axis=0)
    avg_A, avg_B = ep_arr[:,0].mean(), ep_arr[:,1].mean()
    
    writer.close()
    return avg_A, avg_B

In [ ]:
def parameter_grid_search(param_grid, num_episodes = 1000, 
                        warmup = 5000, attempt_rare_transition = 1000,
                        normalize = True, agent_cfgs = agent_cfgs,
                        greedy_rollout = True, num_rollouts = 5
                        ):
    # prepare results table
    results = []

    # base folder for this sweep
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    base_log = os.path.join("runs", f"gridsearch_{ts}")
    os.makedirs(base_log, exist_ok=True)

    # iterate
    for params in ParameterGrid(param_grid):
        # unpack
        lr_actor, lr_critic = params["lr_actor"], params["lr_critic"]
        gamma, tau       = params["gamma"],     params["tau"]
        batch_size, noise_level         = params["batch_size"], params["noise_level"]

        # name this run by its hyper‐params
        run_name = f"firing_mech_lra{lr_actor:.0e}_lrc{lr_critic:.0e}_tau{tau:.0e}_b{batch_size}"
        log_dir  = os.path.join(base_log, run_name)

        # train & get averages
        avg_A, avg_B = training_loop(
            lr_actor=lr_actor, lr_critic=lr_critic,
            gamma=gamma, tau=tau, batch_size=batch_size,
            num_episodes=num_episodes, warmup=warmup, 
            attempt_rare_transition=attempt_rare_transition,
            normalize=normalize,
            agent_cfgs=agent_cfgs,
            greedy_rollout=greedy_rollout,
            num_rollouts=num_rollouts,
            log_dir=log_dir
        )

        # record
        results.append({
            **params,
            "avg_reward_alice": avg_A,
            "avg_reward_bob":   avg_B,
            "log_dir":          log_dir,
        })

    # to DataFrame
    df = pd.DataFrame(results)
    df = df.sort_values("avg_reward_alice", ascending=False)
    df.to_csv(os.path.join(base_log, "grid_results.csv"), index=False)
    return df

# Hyperparameters grid
param_grid = {
    "lr_actor":   [1e-4],
    "lr_critic":  [1e-3], 
    "gamma":      [0.99], 
    "tau":        [0.005],
    "batch_size": [64],
    "noise_level":[0.1] 
}
# Recommended params: lra:1e-4, lrc:1e-3, gam=0.99, tau=0.005, b=64

df = parameter_grid_search(param_grid, num_episodes=1000)

  5%|▌         | 50/1000 [00:14<04:52,  3.25it/s]

Warmup complete, beginning training...


 14%|█▍        | 139/1000 [13:19<50:12,  3.50s/it]  